In [ ]:
# Runs GIZA++ alignments between English and target language Bibles
# Extracts 1-to-1 word alignments
# Reconstructs aligned target-language verses with glosses based on English source
# Outputs verse-level .txt glossed files in this format:
    # verse: <id>
    # ENG: <original English line>
    # XXX: <token1>/<gloss1> token2/gloss2 ...

In [1]:
import os
import re

# Clean up old files
for fname in ['en', 'fr', 'en.vcb', 'fr.vcb', 'en_fr.snt', 'en_fr.cooc',
              'enfr_test.A3.final', 'enfr_test.t3.final', 'enfr_test.dict',
              'config.txt']:
    if os.path.exists(fname):
        os.remove(fname)

In [2]:
# Minimal test data
ce = [['this', 'is', 'a', 'test'], ['another', 'sentence']]
cf = [['ceci', 'est', 'un', 'test'], ['une', 'autre', 'phrase']]

# Write to files
for fn, c in zip(['en', 'fr'], [ce, cf]):
    with open(fn, 'w') as fh:
        fh.write('\n'.join(' '.join(l) for l in c))

In [3]:
giza_path = os.path.expanduser('~/bin/giza-pp/GIZA++-v2/')

In [19]:
from collections import defaultdict

In [4]:
os.system('~/bin/giza-pp/GIZA++-v2/plain2snt.out en fr')

en -> en
fr -> fr


0

In [5]:
os.system('~/bin/giza-pp/GIZA++-v2/snt2cooc.out en.vcb fr.vcb en_fr.snt > en_fr.cooc')

END.


0

In [6]:
with open('config.txt', 'w') as fh:
    fh.write('s: en.vcb\n')
    fh.write('t: fr.vcb\n')
    fh.write('c: en_fr.snt\n')
    fh.write('coocurrencefile: en_fr.cooc\n')
    fh.write('o: enfr_test\n')
    fh.write('v: 1\n')

In [7]:
os.system('~/bin/giza-pp/GIZA++-v2/GIZA++ config.txt')

The following options are from the config file and will be overwritten by any command line options.
Parameter 's' changed from '' to 'en.vcb'
Parameter 't' changed from '' to 'fr.vcb'
Parameter 'c' changed from '' to 'en_fr.snt'
Parameter 'coocurrencefile' changed from '' to 'en_fr.cooc'
Parameter 'o' changed from '2025-05-20.145926.glossophilia' to 'enfr_test'
Parameter 'v' changed from '0' to '1'
general parameters:
-------------------
ml = 101  (maximum sentence length)

No. of iterations:
-------------------
hmmiterations = 5  (mh)
model1iterations = 5  (number of iterations for Model 1)
model2iterations = 0  (number of iterations for Model 2)
model3iterations = 5  (number of iterations for Model 3)
model4iterations = 5  (number of iterations for Model 4)
model5iterations = 0  (number of iterations for Model 5)
model6iterations = 0  (number of iterations for Model 6)

parameter for various heuristics in GIZA++ for efficient training:
------------------------------------------------

Reading vocabulary file from:en.vcb
Reading vocabulary file from:fr.vcb
ERROR: can not read en.vcb.classes
ERROR: can not read fr.vcb.classes
ERROR: can not read en.vcb.classes
ERROR: can not read fr.vcb.classes
ERROR: can not read classes from en.vcb.classes
ERROR: can not read classes from fr.vcb.classes
processing this sentence pair took : 0 seconds
processing this sentence pair took : 0 seconds
NTable contains 80 parameter.
processing this sentence pair took : 0 seconds
processing this sentence pair took : 0 seconds
NTable contains 80 parameter.
processing this sentence pair took : 0 seconds
processing this sentence pair took : 0 seconds
NTable contains 80 parameter.
processing this sentence pair took : 0 seconds
processing this sentence pair took : 0 seconds
NTable contains 80 parameter.
processing this sentence pair took : 0 seconds
processing this sentence pair took : 0 seconds
NTable contains 80 parameter.
processing this sentence pair took : 0 seconds
processing this sentence 

0

In [8]:
def bible_reader_txt(fn):
    for l in open(fn):
        fields = l.strip('\n').split('\t')
        if len(fields) != 2 or re.match('^[0-9]+$', fields[0]) == None:
            continue
        verse = int(fields[0])
        if len(fields[1]) < 5: continue
        yield verse, preprocess(fields[1].lower().strip(' '))
        
def bible_reader_spc(fn):
    for l in open(fn):
        fields = l.strip('\n').split('\t')
        if len(fields) != 2 or re.match('^[0-9]+$', fields[0]) == None:
            continue
        verse = int(fields[0])
        if len(fields[1]) < 5: continue
        yield verse, [(w.split('|')[2] if w.split('|')[2] != '-pron-' else w.split('|')[0])
                      for w in fields[1].lower().strip(' ').split(' ') if w.count('|') > 2]
        
def preprocess(l):
    return [w for w in re.split(u'[\u2014\u0020\-]', l.strip('\n')) if w != '' and w != ' ']

<>:21: SyntaxWarning: invalid escape sequence '\-'
<>:21: SyntaxWarning: invalid escape sequence '\-'
/var/folders/22/4lgkjkkn2vvb2m8ffgsjl7gh0000gn/T/ipykernel_5453/3706348679.py:21: SyntaxWarning: invalid escape sequence '\-'
  return [w for w in re.split(u'[\u2014\u0020\-]', l.strip('\n')) if w != '' and w != ' ']


In [9]:
languages = [
    'kat-x-bible',
    'hin-x-bible',
    'pol-x-bible-nowagdansk',
    'tur-x-bible-goodnews',
    'ind-x-bible-terjemahanbaru',
    'eng-x-bible-newsimplified'
]

base_path = '/Users/glossophilia/Desktop/Data/'

corpora = {
    lg: dict(bible_reader_txt(f'{base_path}{lg}.txt'))
    for lg in languages[:-1]
}

corpora[languages[-1]] = dict(bible_reader_txt(f'{base_path}{languages[-1]}.txt'))

In [10]:
def create_gpp_alignments(lg, ce, cf):
    try:
        os.makedirs(f'alignments_{lg}', exist_ok=True)
    except Exception as e:
        print(f"⚠️ Error creating directory: {e}")
    
    for fn, content in zip(['en', 'fr'], [ce, cf]):
        with open(fn, 'w') as f:
            f.write('\n'.join(' '.join(line) for line in content))

    os.system(f'{giza_path}/plain2snt.out en fr')

    for s, t in [('en', 'fr'), ('fr', 'en')]:
        st = f"{s}_{t}"
        os.system(f'{giza_path}/snt2cooc.out {s}.vcb {t}.vcb {st}.snt > {st}.cooc')

        parameters = {
            'S': f'{s}.vcb',
            'T': f'{t}.vcb',
            'C': f'{st}.snt',
            'CoocurrenceFile': f'{st}.cooc',
            'o': f'{st}',
            'outputpath': f'alignments_{lg}',
        }

        with open('config.txt', 'w') as config_file:
            config_file.write('\n'.join([f'{k}: {v}' for k, v in parameters.items()]))

        os.system(f'{giza_path}/GIZA++ config.txt')

        os.system(f'mv {st}.snt alignments_{lg}/')
        os.system(f'rm {st}.cooc config.txt')

    # Clean up the temporary root files
    os.system('rm en fr en.vcb fr.vcb')


In [11]:
# CREATE GPP ALIGNMENTS
en = languages[-1]

for i, lg in enumerate(languages[:-1]):
    print(f"\n🔄 [{i+1}/{len(languages)-1}] Aligning {lg} ↔ {en}...")
    
    overlap = sorted(set(corpora[lg]) & set(corpora[en]))
    ce = [corpora[en][v] for v in overlap]
    cf = [corpora[lg][v] for v in overlap]
    
    with open(f'{lg}.indices', 'w') as f:
        f.write('\n'.join(map(str, overlap)))
    
    create_gpp_alignments(lg, ce, cf)

    print(f"✅ Alignment complete for {lg}")


🔄 [1/5] Aligning kat-x-bible ↔ eng-x-bible-newsimplified...
en -> en
fr -> fr


line 1000
line 2000
line 3000
line 4000
END.


The following options are from the config file and will be overwritten by any command line options.
Parameter 's' changed from '' to 'en.vcb'
Parameter 't' changed from '' to 'fr.vcb'
Parameter 'c' changed from '' to 'en_fr.snt'
Parameter 'coocurrencefile' changed from '' to 'en_fr.cooc'
Parameter 'o' changed from '2025-05-20.145939.glossophilia' to 'en_fr'
Parameter 'outputpath' changed from '' to 'alignments_kat-x-bible'
general parameters:
-------------------
ml = 101  (maximum sentence length)

No. of iterations:
-------------------
hmmiterations = 5  (mh)
model1iterations = 5  (number of iterations for Model 1)
model2iterations = 0  (number of iterations for Model 2)
model3iterations = 5  (number of iterations for Model 3)
model4iterations = 5  (number of iterations for Model 4)
model5iterations = 0  (number of iterations for Model 5)
model6iterations = 0  (number of iterations for Model 6)

parameter for various heuristics in GIZA++ for efficient training:
-----------------------

Reading vocabulary file from:en.vcb
Reading vocabulary file from:fr.vcb
ERROR: can not read en.vcb.classes
ERROR: can not read fr.vcb.classes


Source vocabulary list has 4879 unique tokens 
Target vocabulary list has 11501 unique tokens 
Calculating vocabulary frequencies from corpus en_fr.snt
Reading more sentence pairs into memory ... 
Corpus fits in memory, corpus has: 4857 sentence pairs.
 Train total # sentence pairs (weighted): 4857
Size of source portion of the training corpus: 116775 tokens
Size of the target portion of the training corpus: 95043 tokens 
In source portion of the training corpus, only 4878 unique tokens appeared
In target portion of the training corpus, only 11499 unique tokens appeared
lambda for PP calculation in IBM-1,IBM-2,HMM:= 95043/(121632-4857)== 0.813899
There are 678219 678219 entries in table
Model1 Training Started at: Tue May 20 14:59:39 2025

-----------
Model1: Iteration 1
Model1: (1) TRAIN CROSS-ENTROPY 13.7682 PERPLEXITY 13951.7
Model1: (1) VITERBI TRAIN CROSS-ENTROPY 18.4762 PERPLEXITY 364666
Model 1 Iteration: 1 took: 0 seconds
-----------
Model1: Iteration 2
Model1: (2) TRAIN CROSS-

...

p0_count is 437002 and p1 is 200194; p0 is 0.68582 p1: 0.31418
Model4: TRAIN CROSS-ENTROPY 5.84937 PERPLEXITY 57.655
Model4: (8) TRAIN VITERBI CROSS-ENTROPY 5.93012 PERPLEXITY 60.9741

Model4 Viterbi Iteration : 8 took: 21 seconds

---------------------
Model4: Iteration 9


10000
20000
30000
NTable contains 166570 parameter.


#centers(pre/hillclimbed/real): 1 1 1  #al: 1205.11 #alsophisticatedcountcollection: 41.4523 #hcsteps: 3.38825
#peggingImprovements: 0
D4 table contains 406 parameters.
A/D table contains 250736 parameters.
A/D table contains 270695 parameters.
p0_count is 445860 and p1 is 195765; p0 is 0.694892 p1: 0.305108
Model4: TRAIN CROSS-ENTROPY 5.80252 PERPLEXITY 55.8126
Model4: (9) TRAIN VITERBI CROSS-ENTROPY 5.88122 PERPLEXITY 58.9417

Model4 Viterbi Iteration : 9 took: 20 seconds

---------------------
Model4: Iteration 10


10000
20000
30000
NTable contains 166570 parameter.
Dumping nTable to: alignments_ind-x-bible-terjemahanbaru/fr_en.n3.final
writing Final tables to Disk 
writing decoder configuration file to alignments_ind-x-bible-terjemahanbaru/fr_en.Decoder.config


#centers(pre/hillclimbed/real): 1 1 1  #al: 1205.74 #alsophisticatedcountcollection: 40.7553 #hcsteps: 3.25239
#peggingImprovements: 0
D4 table contains 406 parameters.
A/D table contains 250736 parameters.
A/D table contains 269971 parameters.
p0_count is 452384 and p1 is 192503; p0 is 0.701494 p1: 0.298506
Model4: TRAIN CROSS-ENTROPY 5.76519 PERPLEXITY 54.3871
Model4: (10) TRAIN VITERBI CROSS-ENTROPY 5.84177 PERPLEXITY 57.3518
Dumping alignment table (a) to file:alignments_ind-x-bible-terjemahanbaru/fr_en.a3.final
Dumping distortion table (d) to file:alignments_ind-x-bible-terjemahanbaru/fr_en.d3.final

Model4 Viterbi Iteration : 10 took: 19 seconds
H3333344444 Training Finished at: Tue May 20 15:12:16 2025


Entire Viterbi H3333344444 Training took: 131 seconds
Writing PERPLEXITY report to: alignments_ind-x-bible-terjemahanbaru/fr_en.perp
Writing source vocabulary list to : alignments_ind-x-bible-terjemahanbaru/fr_en.trn.src.vcb
Writing source vocabulary list to : alignments_ind-x-b

In [22]:
def read_gpp_alignments(lg):
    dirname = f'./alignments_{lg}'
    alignments = defaultdict(list)
    
    paths = {
        'en_fr': os.path.join(dirname, 'en_fr.A3.final'),
        'fr_en': os.path.join(dirname, 'fr_en.A3.final'),
    }

    for direction in ['en_fr', 'fr_en']:
        path = paths[direction]
        if not os.path.exists(path):
            print(f"❌ Missing: {path}")  # Debug print
            raise FileNotFoundError(f"Alignment file not found: {path}")
        else:
            print(f"✅ Found: {path}")  # Debug print
        
        with open(path) as f:
            lines = f.readlines()
        
        for i in range(0, len(lines) - 2, 3):
            tl = lines[i + 2].strip()
            parts = re.split(r'\(\{|\}\)', tl)
            alignments[direction].append([
                (int(k)-1, idx-1) if direction == 'en_fr' else (idx-1, int(k)-1)
                for idx, chunk in enumerate(parts[2::2], start=1)
                if chunk.strip()
                for k in chunk.strip().split()
            ])
    
    return alignments


In [23]:
# Just test one language to start
read_gpp_alignments('hin-x-bible')

✅ Found: ./alignments_hin-x-bible/en_fr.A3.final
✅ Found: ./alignments_hin-x-bible/fr_en.A3.final


defaultdict(list,
            {'en_fr': [[],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
              [],
 

In [24]:
for lg in languages[:-1]:
    print(lg)
    with open('%s_glossed.txt' % lg, 'w') as fo:
        en = languages[-1]
        overlap = sorted(set(corpora[lg]) & set(corpora[en]))
        ce = [corpora[en][o] for o in overlap]
        cf = [corpora[lg][o] for o in overlap]
        try: alignments = read_gpp_alignments(lg)
        except:
            print(lg, 'not found')
            continue
        for i in range(len(overlap)):
            ai_i = set(alignments['en_fr'][i]) & set(alignments['fr_en'][i])
            ai_fe  = alignments['fr_en'][i]
            ai_u   = set(alignments['en_fr'][i]) | set(alignments['fr_en'][i])
            T = defaultdict(lambda : [])
            for fi,f in enumerate(cf[i]):
                t = [ce[i][ei].upper() for ei,fj in ai_i if fj == fi]
                if t == []: t = [ce[i][ei].upper() for ei,fj in ai_fe if fj == fi]
                #if t == []: t = [ce[i][ei].upper() for ei,fj in ai_u if fj == fi]
                T[fi] = t
            fo.write('verse:\t%s\nENG:\t%s\n%s:\t%s\n\n' %
                     (overlap[i], ' '.join(ce[i]), lg[:3].upper(),
                      ' '.join([cf[i][j] + (('/' + '_'.join(T[j])) if j in T else '') 
                                for j in range(len(cf[i]))])))

kat-x-bible
✅ Found: ./alignments_kat-x-bible/en_fr.A3.final
✅ Found: ./alignments_kat-x-bible/fr_en.A3.final
hin-x-bible
✅ Found: ./alignments_hin-x-bible/en_fr.A3.final
✅ Found: ./alignments_hin-x-bible/fr_en.A3.final
pol-x-bible-nowagdansk
✅ Found: ./alignments_pol-x-bible-nowagdansk/en_fr.A3.final
✅ Found: ./alignments_pol-x-bible-nowagdansk/fr_en.A3.final
tur-x-bible-goodnews
✅ Found: ./alignments_tur-x-bible-goodnews/en_fr.A3.final
✅ Found: ./alignments_tur-x-bible-goodnews/fr_en.A3.final
ind-x-bible-terjemahanbaru
✅ Found: ./alignments_ind-x-bible-terjemahanbaru/en_fr.A3.final
✅ Found: ./alignments_ind-x-bible-terjemahanbaru/fr_en.A3.final
